# 锥形传输线锥形线（或简称“锥形器”）是一种阻抗变换器，它通过使用沿线路逐渐变化的特性阻抗 $Z(z)$，将阻抗 $Z_1$ 匹配到阻抗 $Z_2$。可以定义无限多种沿传输线的 $Z(z)$ 曲线。scikit-rf 实现了一个通用的 1D 锥形器 [Taper1D](../../api/taper.rst)，以下直接类由此派生：`Linear`、`Exponential` 和 `SmoothStep`。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf
from skrf.media import MLine
from skrf.network import cascade_list

rf.stylely()

## 几何形状

在本例中，我们将比较一条微带线，其宽度 $W$ 逐渐变化，并串联了一个 15 欧姆电阻：<img src="ANSYS_Circuit_reference.png" width="1000">在比较渐变轮廓之前，让我们定义电路的其他元件，即传输线段和电阻：

In [ ]:
# Model Parameters
from skrf.constants import mil

freq = rf.Frequency(1, 20, unit='GHz', npoints=191)
w1 = 20*mil  # conductor width [m]
w2 = 90*mil  # conductor width [m]
h = 20*mil  # dielectric thickness [m]
t = 0.7*mil  # conductor thickness [m]
rho = 1.724138e-8  # Copper resistivity [Ohm.m]
ep_r = 10  # dielectric relative permittivity
rough = 1e-6  # conductor RMS roughtness [m]
taper_length = 200*mil  # [m]

# Media definitions
microstrip_w1 = MLine(freq, w=w1, h=h, t=t, rho=rho, ep_r=ep_r, rough=rough,
                      disp='kobayashi', z0_port=50)
microstrip_w2 = MLine(freq, w=w2, h=h, t=t, rho=rho, ep_r=ep_r, rough=rough,
                      disp='kobayashi', z0_port=50)

# piece of transmission lines connected to the taper
line1 = microstrip_w1.line(d=50, unit='mil', name='feeder')
line2 = microstrip_w2.line(d=50, unit='mil', name='feeder')

# loading resistor
resistor = microstrip_w2.resistor(R=15)

因此，我们的目标是构建一条特性阻抗可变的传输线。在本示例中，金属层的宽度 $W$ 沿传输线的长度方向变化。最常见的剖面图总结如下：

In [ ]:
fig, ax = plt.subplots()
z = np.linspace(0, taper_length)
ax.plot(z, (w2-w1)/taper_length*z + w1, lw=2, label='linear')
ax.plot(z, w1*np.exp(z/taper_length*(np.log(w2/w1))), lw=2, label='exponential')
ax.set_xticks([0, taper_length])
ax.set_xticklabels(['0', 'taper length'])
ax.set_yticks([w1, w2])
ax.set_yticklabels(['$W_1$', '$W_2$'])
ax.legend()
ax.set_title('Parameter profile along the taper length')

### 线性斜坡

In [ ]:
# create a 2-port Network
taper_linear = rf.taper.Linear(med=MLine, param='w', start=w1, stop=w2,
                        length=taper_length, n_sections=50,
                        med_kw={'frequency': freq, 'h': h, 't':t, 'ep_r': ep_r,
                                'rough': rough, 'rho': rho}).network
print(taper_linear)

# build the full circuit
# equivalent to ntwk = line1 ** taper_linear ** resistor ** line2 ** microstrip_w2.short()
ntwk_linear = cascade_list([line1, taper_linear, line2, resistor, microstrip_w2.short()])

In [ ]:
fig, ax = plt.subplots()
ax.plot(ntwk_linear.frequency.f_scaled, ntwk_linear.s_mag[:,0], lw=2, label='scikit-rf - Linear')

f_ref, s_mag_ref = np.loadtxt('ANSYS_Circuit_taper_linear_s_mag.csv', delimiter=',', skiprows=1, unpack=True)
ax.plot(f_ref, s_mag_ref, label='ANSYS Circuit - Linear Taper', lw=2, ls='--')

ax.set_xlabel('f [GHz]')
ax.set_ylabel('$|s_{11}|$')
ax.set_ylim(0, 0.6)
ax.set_xlim(1, 20)
ax.legend()

### 指数型锥形结构

In [ ]:
taper_exp = rf.taper.Exponential(med=MLine, param='w', start=w1, stop=w2,
                        length=taper_length, n_sections=50,
                        med_kw={'frequency': freq, 'h': h, 't':t, 'ep_r': ep_r,
                                'rough': rough, 'rho': rho}).network

ntwk_exp = line1 ** taper_exp ** line2 ** resistor ** microstrip_w2.short()

In [ ]:
fig, ax = plt.subplots()
ax.plot(ntwk_exp.frequency.f_scaled, ntwk_exp.s_mag[:,0], lw=2, label='scikit-rf - Exponential')

f_ref, s_mag_ref = np.loadtxt('ANSYS_Circuit_taper_exponential_s_mag.csv', delimiter=',', skiprows=1, unpack=True)
ax.plot(f_ref, s_mag_ref, label='ANSYS Circuit - Exponential Taper', lw=2, ls='--')

ax.set_xlabel('f [GHz]')
ax.set_ylabel('$|s_{11}|$')
ax.set_ylim(0, 0.6)
ax.set_xlim(1, 20)
ax.legend()